# DreamerV3 config → sampling and update budget (RLlib new API stack)

Companion to `sac_training_intensity.ipynb`. Where SAC turns `training_intensity`
into two round-robin weights, DreamerV3 has no rr-weights at all: it runs two
`while` loops driven by `training_ratio`, and its batch size is `batch_size_B x
batch_length_T` rather than `total_train_batch_size`.

This notebook derives, from the config fields, how many env steps are sampled,
how many gradient updates run, and how many replayed real transitions and
imagined (world-model-generated) transitions each gradient update consumes.
Section 0 pins down exactly what each of those counted units is; every table
column is defined in terms of them. Section 7 cross-checks the derivations
against a real 14,000-iteration run from this repo.

**Scope: new API stack only** — DreamerV3 forces it in its own constructor
(`ray/rllib/algorithms/dreamerv3/dreamerv3.py:143-145`, `enable_rl_module_and_learner
= enable_env_runner_and_connector_v2 = True`).

### Where this lives in RLlib (Ray 2.52.1)

`DreamerV3.training_step()` (`dreamerv3.py:499`) is one sampling loop followed by
one update loop:

```python
have_sampled = False
while (                                                            # dreamerv3.py:513
    self.replay_buffer.get_num_timesteps()
    < (self.config.batch_size_B * self.config.batch_length_T)      # a) buffer too small (:516-517)
    or self.training_ratio >= self.config.training_ratio           # b) trained enough already (:520)
    or not have_sampled                                            # c) at least one round (:522)
):
    episodes, env_runner_results = synchronous_parallel_sample(
        worker_set=self.env_runner_group,
        max_agent_steps=(self.config.rollout_fragment_length
                         * self.config.num_envs_per_env_runner),   # dreamerv3.py:525-531
        ...)
    self.replay_buffer.add(episodes=episodes)                      # dreamerv3.py:540
    have_sampled = True
    env_steps_last_regular_sample = sum(len(eps) for eps in episodes)   # dreamerv3.py:544

replayed_steps_this_iter = sub_iter = 0
while (replayed_steps_this_iter
       / env_steps_last_regular_sample) < self.config.training_ratio:   # dreamerv3.py:589-591
    sample = self.replay_buffer.sample(batch_size_B=..., batch_length_T=...)  # :597
    replayed_steps_this_iter += self.config.batch_size_B * self.config.batch_length_T  # :601-602
    learner_results = self.learner_group.update(batch=...)                    # :614
```

Three consequences the tables below quantify:

1. The update loop divides by `env_steps_last_regular_sample` — the **last**
   sampling round only, not everything sampled this iteration (it is reassigned
   inside the sampling loop at `dreamerv3.py:544`).
2. It is a do-while: `replayed_steps_this_iter` starts at `0`, so every
   `training_step()` runs **at least one** update of `B x T` replayed steps,
   however few env steps were collected.
3. Sampling-loop condition (b) reads the `DreamerV3.training_ratio` property,
   which is **inert on Ray 2.52.1** — it always returns `0.0` because it peeks
   a metrics key that is never written (details in section 0, empirical proof
   in section 7). Post-warm-up, every `training_step()` therefore samples
   **exactly one** round, and the ratio is only enforced per-iteration by the
   update loop.

## 0. Terminology — what each counted unit is

Every number in this notebook counts one of the following units. All line
references are Ray 2.52.1 unless prefixed with a repo path.

* **env step** — one real `env.step()` executed by an EnvRunner in the actual
  `AdvBuildingGym` environment, i.e. one 5-minute control step of genuinely new
  experience. This is the unit of `NUM_ENV_STEPS_SAMPLED(_LIFETIME)` and of the
  x-axis / stop criterion `num_env_steps_sampled_lifetime`.

* **sampling round** — one `synchronous_parallel_sample()` call inside the
  sampling loop (`dreamerv3.py:525-541`). It collects
  `sampling_runners x rollout_fragment_length x num_envs_per_env_runner` env
  steps (section 2) and appends them to the `EpisodeReplayBuffer` as episode
  chunks. The sampling loop *can* run several rounds back to back (conditions
  a–c in the listing above), but on Ray 2.52.1 condition (b) is inert (see
  `training_ratio` below), so extra rounds only happen during the initial
  buffer warm-up via condition (a).

* **`training_step()` call = one reported iteration ("iter")** — one execution
  of `DreamerV3.training_step()`: one sampling loop followed by one update
  loop. The `min_time_s_per_iteration` / `min_sample_timesteps_per_iteration` /
  `min_train_timesteps_per_iteration` knobs that could bundle several
  `training_step()` calls into one reported result default to `None` / `0` / `0`
  (`ray/rllib/algorithms/algorithm_config.py:566-568`) and are overridden
  neither by `DreamerV3Config` nor by this repo — so one Tune result iteration
  is exactly one `training_step()` call.

* **gradient update ("update")** — one pass through the update loop = one
  `self.learner_group.update(batch)` call on one `B x T` batch
  (`dreamerv3.py:597-614`). Internally that is exactly **three backward passes
  and three Adam steps on disjoint parameter sets** — world model, actor,
  critic — executed by the learner's per-component gradient computation
  (`torch/dreamerv3_torch_learner.py:48-91` registers the three optimizers;
  `:148-178` runs one `loss.backward()` per component and zeroes cross-talk
  into the world model). So "6 updates/iter" means: 6 world-model steps + 6
  actor steps + 6 critic steps, all from 6 replayed batches. Counted in
  `num_grad_updates_lifetime` (+1 per update, `dreamerv3.py:630`).

* **replayed real step** — one env transition **drawn back out of the replay
  buffer** as part of a `B x T` train batch: `batch_size_B` independent
  sequences of `batch_length_T` *consecutive* stored timesteps
  (`dreamerv3.py:597`). Three properties matter:
  1. it is *real* experience (originally produced by the env, not by the model);
  2. it is counted **with multiplicity** — a stored transition sampled into
     two different batches counts twice; "replayed steps" is a training-volume
     measure, not a count of unique transitions;
  3. only the **world-model** losses (decoder / reward / continue / dynamics /
     representation) consume it directly
     (`torch/dreamerv3_torch_learner.py:191-201`).
  The update loop books `B x T` of these per update into its loop counter
  `replayed_steps_this_iter` (`dreamerv3.py:601-602`). **Caution:** the
  *metric* `NUM_ENV_STEPS_TRAINED(_LIFETIME)` under-counts this by a factor of
  `T` — the Learner logs `batch.env_steps()` per update
  (`core/learner/learner.py:1710-1721`), and for the `(B, T)`-shaped replay
  batch `env_steps()` is `len(batch)` = the leading dimension `B`, not
  `B x T` (`policy/sample_batch.py:281-286`). Section 7 shows this on a real
  run (192 = 12 updates x B=16 per iteration, not 12 x 256).

* **imagined step (dreamed step)** — one latent-space transition generated by
  the world model's own dynamics via `dreamer_model.dream_trajectory()`
  (`torch/dreamerv3_torch_learner.py:282-294`): starting from the posterior
  state `(h, z)` of **every** one of the `B x T` replayed timesteps, the model
  rolls forward `horizon_H` steps *without touching the env or the buffer*,
  with actions chosen by the current actor
  (`torch/models/dreamer_model.py:205-207`). Hence `B x T x H` imagined steps
  per update. They are the **only** data the actor and critic losses see
  (`torch/dreamerv3_torch_learner.py:331-345` — both losses take `dream_data`,
  never `batch`), and they appear in **no** sampled/trained step counter — they
  are pure compute, invisible to `training_ratio`.

* **`training_ratio`** — the configured target for
  `replayed real steps / env steps sampled`, i.e. how many buffer-drawn
  transitions the world model trains on per fresh env step. The *intended*
  enforcement is global over the run: the `DreamerV3.training_ratio` property
  (`dreamerv3.py:689-706`) is supposed to divide lifetime trained steps by
  lifetime sampled steps, and sampling-loop condition (b) samples extra rounds
  while that quotient is still at or above target. **On Ray 2.52.1 this
  mechanism is broken**: the property peeks the *top-level* metrics key
  `num_env_steps_trained_lifetime` with `default=0`, but the only writer logs
  that counter under `(LEARNER_RESULTS, ALL_MODULES, ...)`
  (`core/learner/learner.py:1716-1721`; nothing in `rllib/` writes the
  top-level key), so the property always returns `0.0` — as does the logged
  `actual_training_ratio` metric (`dreamerv3.py:687`). Net effect: condition
  (b) never fires after warm-up, each iteration samples exactly one round, and
  the ratio is enforced only *per iteration* by the update loop's
  `replayed_steps_this_iter` counter. Section 7 verifies all of this on a real
  run. Imagined steps never enter this ratio in any variant.

## 1. Config parameters

`DreamerV3Config` deliberately leaves the generic batch machinery unset —
`train_batch_size = None` with the comment *"Do not use! Set `batch_size_B` and
`batch_length_T` instead"* (`dreamerv3.py:139-140`). Reading
`config.train_batch_size_per_learner` on a `DreamerV3Config` actually raises
`TypeError` (`None // 1`; reproduced on the installed Ray 2.52.1), so
`total_train_batch_size` plays no role here; the train batch is
`batch_size_B x batch_length_T` transitions.

The values below are **this notebook's working values** for the sweep. The
repo's *defaults* come from
`adv_building_gym/config/training/training_param_config.py:95-103` (applied in
`adv_building_gym/ray/training/select_model.py:103-123`) and keep RLlib's
`B=16 / T=64 / H=15`, changing only `training_ratio=10.0` and the replay
sizing; individual trial YAMLs override these per run (e.g. the section-7
reference run uses `B=16, T=16, H=15`). Trailing comments give the RLlib
default and the repo default per line.

In [ ]:
import math

import pandas as pd

# Section 5 imports RLlib's DreamerV3 package, which pulls in Gymnasium's Box /
# passive-env-checker warnings; reuse the repo's own filters for those.
from adv_building_gym._common.warning_filters import setup_warning_filters

setup_warning_filters()

# --- DreamerV3 config fields ----------------------------------------------
# Trailing comments give the DreamerV3Config() default (verified on Ray 2.52.1,
# dreamerv3.py:100-112) and the repo default from TrainingParamConfig (:95-103).
MODEL_SIZE = "XS"          # config.model_size — RLlib default: "XS"; repo default: "XS"
TRAINING_RATIO = 10.0      # config.training_ratio — RLlib default: 1024; repo default: 10.0
BATCH_SIZE_B = 64          # config.batch_size_B — RLlib & repo default: 16 (notebook working value)
BATCH_LENGTH_T = 8        # config.batch_length_T — RLlib & repo default: 64 (notebook working value)
HORIZON_H = 15             # config.horizon_H (imagination horizon) — RLlib & repo default: 15 (notebook working value)

ROLLOUT_FRAGMENT_LENGTH = 288  # config.rollout_fragment_length — DreamerV3 default: 1 (dreamerv3.py:142)
#   The repo leaves it at the default unless dreamerv3.rollout_length is set (select_model.py:124-127).
NUM_ENVS_PER_ENV_RUNNER = 1  # config.num_envs_per_env_runner — RLlib default: 1
NUM_ENV_RUNNERS = 0          # config.num_env_runners — DreamerV3 default: 0 (dreamerv3.py:141);
#   the repo derives it from the SLURM CPU count in common_model_config.py.

# Replay buffer: repo sets capacity = episode_length * episodes_to_keep
# (select_model.py:118-122); RLlib default capacity is int(1e6) (dreamerv3.py:105).
EPISODE_LENGTH = 288                    # env steps per episode (5-min control step, one day)
EPISODES_TO_KEEP_IN_REPLAY_BUFFER = 500  # repo default (TrainingParamConfig:100)

# --- Sweep -----------------------------------------------------------------
TRAINING_RATIOS = [1, 2, 5, 10, 32, 64, 128, 256, 512, 1024]
HIGHLIGHT_RATIO = TRAINING_RATIO   # row to mark in the table

## 2. Sampling: env steps per round

`synchronous_parallel_sample` calls `sample()` on every runner with no step
argument (`ray/rllib/execution/rollout_ops.py:105-115`), so each runner returns
`get_rollout_fragment_length(worker_index) * num_envs` timesteps
(`ray/rllib/env/single_agent_env_runner.py:217-226`). `max_agent_steps` only
stops the *while* loop, it does not truncate a round — so a round collects the
full `N x rollout_fragment_length x num_envs` even though DreamerV3 caps at
`rollout_fragment_length x num_envs` (`dreamerv3.py:527-531`).

With `num_env_runners > 0` only the remote runners sample
(`local_env_runner=False`, `rollout_ops.py:105-115`); with `0` the local runner
is used instead (`rollout_ops.py:101-103`). Note also that `DreamerV3Config`
does **not** override `get_rollout_fragment_length`, and its default is the int
`1` (not `"auto"`), so the value is used verbatim.

> The in-source comment at `dreamerv3.py:146-148` claims DreamerV3 "still uses
> its own EnvRunner class". That is stale: `env_runner_cls` is `None` on a built
> `DreamerV3Config`, and `EnvRunnerGroup` therefore picks `SingleAgentEnvRunner`
> (`ray/rllib/env/env_runner_group.py:147-176`).

In [ ]:
def env_steps_per_sampling_round():
    """Env steps actually collected per `synchronous_parallel_sample` call."""
    sampling_runners = NUM_ENV_RUNNERS if NUM_ENV_RUNNERS > 0 else 1  # local runner if no remotes
    return sampling_runners * ROLLOUT_FRAGMENT_LENGTH * NUM_ENVS_PER_ENV_RUNNER


def train_batch_timesteps():
    """Replayed real transitions per gradient update: B x T (dreamerv3.py:601)."""
    return BATCH_SIZE_B * BATCH_LENGTH_T


steps_per_round = env_steps_per_sampling_round()
batch_timesteps = train_batch_timesteps()
imagined_per_update = batch_timesteps * HORIZON_H  # each replayed timestep seeds one length-H dream
buffer_capacity = EPISODE_LENGTH * EPISODES_TO_KEEP_IN_REPLAY_BUFFER

# dtype=object keeps ints from being widened to float by the ratio entry.
summary = pd.Series(
    {
        "model_size": MODEL_SIZE,
        "training_ratio (configured)": TRAINING_RATIO,
        "batch_size_B": BATCH_SIZE_B,
        "batch_length_T": BATCH_LENGTH_T,
        "horizon_H": HORIZON_H,
        # Steps consumed by ONE gradient update (one learner_group.update() call, section 0):
        "replayed real steps / grad update (B x T)": batch_timesteps,
        "imagined steps / grad update (B x T x H)": imagined_per_update,
        "steps touched / grad update (B x T x (1 + H))": batch_timesteps + imagined_per_update,
        "rollout_fragment_length": ROLLOUT_FRAGMENT_LENGTH,
        "num_envs_per_env_runner": NUM_ENVS_PER_ENV_RUNNER,
        "num_env_runners": NUM_ENV_RUNNERS,
        "env steps / sampling round": steps_per_round,
        "replay capacity (timesteps)": buffer_capacity,
        "replay capacity (episodes)": EPISODES_TO_KEEP_IN_REPLAY_BUFFER,
        "min buffer fill before training (B x T)": batch_timesteps,
    },
    name="value",
    dtype=object,
)
summary.to_frame().style.set_table_styles(
    [{"selector": "td", "props": [("text-align", "right"), ("padding", "2px 10px")]}]
)

,value
model_size,XS
training_ratio (configured),10.000000
batch_size_B,64
batch_length_T,8
horizon_H,6
replayed real steps / grad update (B x T),512
imagined steps / grad update (B x T x H),3072
steps touched / grad update (B x T x (1 + H)),3584
rollout_fragment_length,288
num_envs_per_env_runner,1


## 3. Update budget per `training_step()`

The update loop (`dreamerv3.py:589-591`) is a do-while over
`replayed_steps_this_iter / env_steps_last_regular_sample < training_ratio`,
adding `B x T` per pass, so

```
gradient updates = max(1, ceil(training_ratio * env_steps_last_regular_sample / (B * T)))
```

One **gradient update** (one `learner_group.update()` call — one Adam step each
for world model, actor, and critic; see section 0) consumes exactly:

* **`B x T` replayed real transitions** — `batch_size_B` sequences of
  `batch_length_T` consecutive env transitions drawn from the
  `EpisodeReplayBuffer` (`dreamerv3.py:597`), counted with multiplicity. They
  are the training data of the **world-model losses only**
  (`torch/dreamerv3_torch_learner.py:191-201`) and are what the update loop
  books against `training_ratio`.
* **`B x T x H` imagined transitions** — latent rollouts the world model
  generates from the posterior state of **every** replayed timestep in the
  batch, `horizon_H` steps each, actions picked by the current actor
  (`torch/dreamerv3_torch_learner.py:277-294`: *"we are starting a new dream
  trajectory at every actually encountered timestep in the batch, so we are
  creating B*T trajectories of len horizon_H"*). They are the training data of
  the **actor and critic losses only** (`:331-345`) and count toward no step
  metric. The dreamed tensors carry `H + 1` entries — the start state plus `H`
  dreamed steps (`torch/models/dreamer_model.py:258-277`: the dreamed reward /
  continue tensors are reshaped to `[timesteps_H + 1, -1]`).

In total one gradient update therefore touches `B x T x (1 + H)` transitions.
The section-2 summary table quantifies all three counts (`B x T`, `B x T x H`,
`B x T x (1 + H)`) with this notebook's configured values.

In [ ]:
def updates_per_training_step(training_ratio, env_steps_last_round):
    """Gradient updates (= learner_group.update() calls) per training_step (dreamerv3.py:589-591)."""
    updates = math.ceil(training_ratio * env_steps_last_round / train_batch_timesteps())
    return max(1, updates)  # do-while: replayed starts at 0, so >= 1 update always runs


def env_steps_per_update_at_target(training_ratio):
    """Env steps that must be sampled to 'pay' for one B x T update at the target ratio."""
    return train_batch_timesteps() / training_ratio

## 4. `training_ratio` sweep

Per candidate `training_ratio`, assuming one sampling round happened last
(`env_steps_last_regular_sample = env steps / sampling round` from section 2).
Column definitions, in the units of section 0:

| column | exact meaning |
| --- | --- |
| `training_ratio` | the candidate target: replayed real steps per env step sampled |
| `env steps/iter` | fresh env transitions collected by the one sampling round of this `training_step()`: `sampling_runners x rollout_fragment_length x num_envs_per_env_runner` |
| `grad updates/iter` | `learner_group.update()` calls in this `training_step()` — each is one Adam step for world model + actor + critic: `max(1, ceil(ratio x env steps / (B x T)))` |
| `replayed real steps/iter` | env transitions drawn back out of the replay buffer this `training_step()` (with multiplicity): `grad updates x B x T`. World-model training volume; what the update loop counts in `replayed_steps_this_iter`. (The `NUM_ENV_STEPS_TRAINED` *metric* records only `grad updates x B` — see section 0.) |
| `imagined steps/iter` | latent transitions the world model dreams this `training_step()`: `grad updates x B x T x H`. Actor/critic training volume; booked in no step counter |
| `target r:s` | the candidate `training_ratio` restated as a float (replayed real steps : env steps sampled) |
| `realized r:s` | `replayed real steps/iter ÷ env steps/iter` — the ratio actually achieved per iteration. It overshoots the target whenever the `ceil` / `max(1, …)` floor forces a whole extra `B x T` batch (visible for small ratios) |
| `env steps/update` | steady-state price of one gradient update: env steps that must be sampled so one `B x T` update keeps the ratio at target, `B x T ÷ ratio` |
| `rounds/update` | the same price in sampling rounds: `env steps/update ÷ env steps/round`. Values < 1 mean a single round funds several updates |

On Ray 2.52.1, `realized r:s` is not only the per-iteration figure but also the
**long-run** figure: sampling-loop condition (b) (`dreamerv3.py:520`), which was
meant to pull the lifetime ratio back to target by taking extra sampling
rounds, compares against the `DreamerV3.training_ratio` property — and that
property is constantly `0.0` (it peeks a metrics key that is never written; see
section 0). Post-warm-up every iteration samples exactly one round, so the
small-ratio overshoot from the `ceil` / `max(1, …)` floor is **permanent**, not
transient. Section 7 confirms this on a real run.

In [ ]:
rows = []
for ratio in TRAINING_RATIOS:
    updates = updates_per_training_step(ratio, steps_per_round)
    replayed = updates * batch_timesteps
    imagined = replayed * HORIZON_H
    steps_per_update = env_steps_per_update_at_target(ratio)

    rows.append(
        {
            "training_ratio": ratio,
            "env steps/iter": steps_per_round,
            "grad updates/iter": updates,
            "replayed real steps/iter": replayed,
            "imagined steps/iter": imagined,
            "target r:s": float(ratio),
            "realized r:s": replayed / steps_per_round,
            "env steps/update": steps_per_update,
            "rounds/update": steps_per_update / steps_per_round,
        }
    )

sweep = pd.DataFrame(rows)
sweep

,training_ratio,env steps/iter,grad updates/iter,replayed real steps/iter,imagined steps/iter,target r:s,realized r:s,env steps/update,rounds/update
0,1,288,1,512,3072,1.0,1.777778,512.0,1.777778
1,2,288,2,1024,6144,2.0,3.555556,256.0,0.888889
2,5,288,3,1536,9216,5.0,5.333333,102.4,0.355556
3,10,288,6,3072,18432,10.0,10.666667,51.2,0.177778
4,32,288,18,9216,55296,32.0,32.000000,16.0,0.055556
5,64,288,36,18432,110592,64.0,64.000000,8.0,0.027778
6,128,288,72,36864,221184,128.0,128.000000,4.0,0.013889
7,256,288,144,73728,442368,256.0,256.000000,2.0,0.006944
8,512,288,288,147456,884736,512.0,512.000000,1.0,0.003472
9,1024,288,576,294912,1769472,1024.0,1024.000000,0.5,0.001736


### Formatted view

The highlighted row is `HIGHLIGHT_RATIO` (this repo's configured value).

In [ ]:
def highlight_configured(row):
    is_match = row["training_ratio"] == HIGHLIGHT_RATIO
    return ["font-weight: bold; background-color: rgba(255, 214, 102, 0.35)" if is_match else ""] * len(row)


styled = (
    sweep.style.hide(axis="index")
    .format(
        {
            "env steps/iter": "{:,d}",
            "grad updates/iter": "{:,d}",
            "replayed real steps/iter": "{:,d}",
            "imagined steps/iter": "{:,d}",
            "target r:s": "{:.1f}",
            "realized r:s": "{:.1f}",
            "env steps/update": "{:.2f}",
            "rounds/update": "{:.2f}",
        }
    )
    .apply(highlight_configured, axis=1)
    .set_caption(
        f"DreamerV3 update budget — B={BATCH_SIZE_B}, T={BATCH_LENGTH_T}, "
        f"BxT={batch_timesteps}, H={HORIZON_H}, rollout={ROLLOUT_FRAGMENT_LENGTH}, "
        f"envs/runner={NUM_ENVS_PER_ENV_RUNNER}, runners={NUM_ENV_RUNNERS} "
        f"-> {steps_per_round} env steps/round"
    )
    .set_table_styles(
        [
            {"selector": "caption", "props": [("caption-side", "top"), ("font-size", "0.95em"), ("padding-bottom", "0.5em")]},
            {"selector": "th", "props": [("text-align", "right"), ("padding", "2px 10px")]},
            {"selector": "td", "props": [("text-align", "right"), ("padding", "2px 10px")]},
        ]
    )
)
styled

training_ratio,env steps/iter,grad updates/iter,replayed real steps/iter,imagined steps/iter,target r:s,realized r:s,env steps/update,rounds/update
1,288,1,512,"3,072",1.0,1.8,512.00,1.78
2,288,2,"1,024","6,144",2.0,3.6,256.00,0.89
5,288,3,"1,536","9,216",5.0,5.3,102.40,0.36
10,288,6,"3,072","18,432",10.0,10.7,51.20,0.18
32,288,18,"9,216","55,296",32.0,32.0,16.00,0.06
64,288,36,"18,432","110,592",64.0,64.0,8.00,0.03
128,288,72,"36,864","221,184",128.0,128.0,4.00,0.01
256,288,144,"73,728","442,368",256.0,256.0,2.00,0.01
512,288,288,"147,456","884,736",512.0,512.0,1.00,0.00
1024,288,576,"294,912","1,769,472",1024.0,1024.0,0.50,0.00


### Plain-text view

Same table without the HTML styler, for terminals and log files.

In [ ]:
plain = sweep.copy()
for column, spec in {
    "target r:s": "{:.1f}",
    "realized r:s": "{:.1f}",
    "env steps/update": "{:.2f}",
    "rounds/update": "{:.2f}",
}.items():
    plain[column] = plain[column].map(spec.format)

print(plain.to_string(index=False))

 training_ratio  env steps/iter  grad updates/iter  replayed real steps/iter  imagined steps/iter target r:s realized r:s env steps/update rounds/update
              1             288                  1                       512                 3072        1.0          1.8           512.00          1.78
              2             288                  2                      1024                 6144        2.0          3.6           256.00          0.89
              5             288                  3                      1536                 9216        5.0          5.3           102.40          0.36
             10             288                  6                      3072                18432       10.0         10.7            51.20          0.18
             32             288                 18                      9216                55296       32.0         32.0            16.00          0.06
             64             288                 36                     18432      

## 5. What `model_size` selects

`model_size` is a single switch over the network dimensions in table B of the
DreamerV3 paper. The numbers below are read straight out of RLlib's lookup
functions (`ray/rllib/algorithms/dreamerv3/utils/__init__.py:24-155`) rather than
copied, so they cannot drift from the installed version.

Only `dense_hidden_units`, `num_dense_layers`, `gru_units`, and the
`z_categoricals x z_classes` latent shape matter for this repo — the CNN
multiplier applies to image observations, and `AdvBuildingGym` observations are
flat `Box` vectors.

Link: https://arxiv.org/pdf/2301.04104 (table B)

In [ ]:
from ray.rllib.algorithms.dreamerv3.utils import (
    _ALLOWED_MODEL_DIMS,
    get_cnn_multiplier,
    get_dense_hidden_units,
    get_gru_units,
    get_num_dense_layers,
    get_num_z_categoricals,
    get_num_z_classes,
)

model_sizes = pd.DataFrame(
    [
        {
            "model_size": size,
            "dense_hidden_units": get_dense_hidden_units(size),
            "num_dense_layers": get_num_dense_layers(size),
            "gru_units": get_gru_units(size),
            "z_categoricals": get_num_z_categoricals(size),
            "z_classes": get_num_z_classes(size),
            "latent z dims": get_num_z_categoricals(size) * get_num_z_classes(size),
            "cnn_multiplier (image obs only)": get_cnn_multiplier(size),
        }
        for size in _ALLOWED_MODEL_DIMS
    ]
)


def highlight_model_size(row):
    is_match = row["model_size"] == MODEL_SIZE
    return ["font-weight: bold; background-color: rgba(255, 214, 102, 0.35)" if is_match else ""] * len(row)


(
    model_sizes.style.hide(axis="index")
    .apply(highlight_model_size, axis=1)
    .set_caption(
        "model_size presets — first four are RLlib debug sizes not listed in the paper; "
        f"configured: {MODEL_SIZE}"
    )
    .set_table_styles(
        [
            {"selector": "caption", "props": [("caption-side", "top"), ("font-size", "0.95em"), ("padding-bottom", "0.5em")]},
            {"selector": "th", "props": [("text-align", "right"), ("padding", "2px 10px")]},
            {"selector": "td", "props": [("text-align", "right"), ("padding", "2px 10px")]},
        ]
    )
)

model_size,dense_hidden_units,num_dense_layers,gru_units,z_categoricals,z_classes,latent z dims,cnn_multiplier (image obs only)
nano,16,1,16,4,4,16,2
micro,32,1,32,8,8,64,4
mini,64,1,64,16,16,256,8
XXS,128,1,128,32,32,1024,16
XS,256,1,256,32,32,1024,24
S,512,2,512,32,32,1024,32
M,640,3,1024,32,32,1024,48
L,768,4,2048,32,32,1024,64
XL,1024,5,4096,32,32,1024,96


## 6. Reading the tables

* **`training_ratio` is replayed-real-steps-per-env-step**, the same units as
  SAC's `training_intensity` — but it is enforced by a loop condition rather
  than a rounded round-robin weight. The *designed* enforcement is global
  (lifetime trained ÷ lifetime sampled, `DreamerV3.training_ratio` property,
  `dreamerv3.py:689-706`), but on Ray 2.52.1 that property is inert (always
  `0.0`; see section 0), so the effective enforcement is the update loop's
  per-iteration counter only. Imagined steps are excluded from the ratio in
  either variant.
* **`B x T` is the update quantum.** One gradient update always replays exactly
  `B x T` timesteps (1024 at the default `16 x 64`), so a small
  `training_ratio` cannot buy a smaller update — it buys *fewer* gradient
  updates, and the realized ratio overshoots whenever
  `ratio x env steps < B x T` (the `max(1, ...)` floor in section 3) —
  permanently so, given the inert condition (b).
* **`horizon_H` multiplies actor/critic data, not the ratio.** Every replayed
  timestep seeds one dream of length `H`, so imagined volume is always exactly
  `H x` the replayed volume — raising `H` raises actor/critic compute and
  bootstrap length per update but changes neither the number of updates nor
  the training ratio.
* **RLlib's defaults are self-consistent**: `training_ratio=1024` with
  `B x T = 1024` and `rollout_fragment_length=1`, `num_env_runners=0` means one
  env step then exactly one gradient update — realized ratio 1024. Raising
  `num_env_runners` above 0 multiplies the env steps per round and thus changes
  the sampling/update rhythm.
* **The replay buffer must hold `B x T` timesteps before any training happens**
  (sampling-loop condition (a), `dreamerv3.py:513-517`), and on the very first
  iteration the gap is filled with **random** actions
  (`dreamerv3.py:546-568`) — `random_actions=True`, capped at
  `B x T - env_steps_last_regular_sample`.
* **Repo replay capacity is `EPISODE_LENGTH x episodes_to_keep`** (144,000
  timesteps at 288 x 500), far below RLlib's `int(1e6)` default — chosen so the
  buffer holds a bounded number of whole episodes.

## 7. Empirical check against a real run (Ray 2.52.1)

Reference run: `snapshots/20260718_140021_gen_battery_lin_capacity_sampled_dreamerv3_long`,
14,000 reported iterations. Its `params.json` confirms the effective config:
`batch_size_B=16`, `batch_length_T=16` (→ `B x T = 256`), `training_ratio=10.0`,
`horizon_H=15`, `rollout_fragment_length=288`, `num_envs_per_env_runner=1`,
`num_env_runners=0` (→ the local runner samples, 288 env steps per round).

Every derivation in this notebook checks out against the final `result.json`
line — and the two Ray 2.52.1 metric defects are visible exactly as predicted:

| quantity | predicted | observed (iter 14,000) |
| --- | --- | --- |
| env steps / iter | one round = `1 x 288 x 1 = 288` (condition (b) inert → never an extra round) | `num_env_steps_sampled_lifetime = 4,032,000 = 14,000 x 288` exactly |
| grad updates / iter | `max(1, ceil(10 x 288 / 256)) = 12` | `num_grad_updates_lifetime = 168,000 = 14,000 x 12` exactly |
| replayed real steps / iter | `12 x 256 = 3,072` (realized r:s `= 10.67`, permanent overshoot over target 10) | not directly logged — see next row |
| `NUM_ENV_STEPS_TRAINED_LIFETIME` metric | under-counts by factor `T`: `12 x B = 192`/iter | `learners/__all_modules__/num_env_steps_trained_lifetime = 2,688,000 = 14,000 x 192` exactly |
| `actual_training_ratio` metric | `0.0` (property peeks a never-written top-level key) | `0.0` at iteration 1 **and** at iteration 14,000 |

Readings:

* The **12-updates prediction matching `num_grad_updates_lifetime` exactly**
  validates the update-budget formula of section 3 (including the `ceil`).
* `4,032,000 = 14,000 x 288` sampled steps means **not a single extra sampling
  round** was ever taken beyond the one per iteration — condition (b) never
  fired, as the broken `training_ratio` property predicts. (The warm-up round
  needed no top-up either: 288 ≥ `B x T = 256`.)
* The true realized ratio of this run is `3,072 / 288 = 10.67` replayed real
  steps per env step — the sweep-table `realized r:s` value, held for the whole
  run. Neither logged metric shows it: `actual_training_ratio` reads 0.0, and
  reconstructing it from `num_env_steps_trained_lifetime` requires multiplying
  by `T` first (`2,688,000 x 16 / 4,032,000 = 10.67`).